# 04 — Phase 3: ECG + Metadata Fusion

This notebook trains a late-fusion model using the ECG waveform and **age/sex only**. It does not use SCP codes or diagnosis-derived report text because those can reveal the target label.

Checkpoint selection uses a held-out validation split from site 1. Site 2 is evaluated only after selecting the checkpoint. Height and weight are excluded from this primary experiment because they are missing in 98% of site-2 records.

# 04 — Multimodal Fusion Experiment

## Objective
Train and evaluate a model that combines ECG signals with patient metadata (age and sex).

## Experimental setup
- Dataset: PTB-XL
- Source/training site: Site 1
- Held-out evaluation site: Site 2
- Signal-only baseline: `baseline_signal_only_site1v2`
- Fusion experiment: `fusion_metadata_site1v2`
- Rare class of interest: HYP
- Training epochs: 20
- Hardware: NVIDIA T4 GPU


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess

REPO_URL = "https://github.com/Sakina152/mm-dg-healthcare.git"
REPO_DIR = "/content/mm-dg-healthcare"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!git log -1 --oneline

Mounted at /content/drive
Working directory: /content/mm-dg-healthcare
dc3eab7 (HEAD -> main, origin/main, origin/HEAD) Save rare class identification analysis


## 1. Environment verification

Before training, verify that the GPU, fusion configuration, trainer, and dataset are available. Do not begin training until all required checks pass.

In [2]:
import os
import torch

os.chdir("/content/mm-dg-healthcare")

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Fusion config:",
      os.path.exists("configs/fusion_metadata.yaml"))
print("Fusion trainer:",
      os.path.exists("src/training/trainer_fusion.py"))
print("PTB-XL database:",
      os.path.exists("data/raw/ptbxl/ptbxl_database.csv"))
print("PTB-XL signal folder:",
      os.path.isdir("data/raw/ptbxl/records100"))

assert torch.cuda.is_available(), "Switch to a T4 GPU first."
assert os.path.exists("configs/fusion_metadata.yaml"), "Fusion config missing."
assert os.path.exists("src/training/trainer_fusion.py"), "Trainer missing."

GPU available: True
GPU: Tesla T4
Fusion config: True
Fusion trainer: True
PTB-XL database: False
PTB-XL signal folder: False


In [3]:
archive = "/content/drive/MyDrive/mm-dg-healthcare-data/ptbxl.tar"

print("Archive exists:", os.path.exists(archive))
if os.path.exists(archive):
    print("Archive size (GB):",
          round(os.path.getsize(archive) / 1e9, 2))

Archive exists: True
Archive size (GB): 0.58


In [4]:
import os
import tarfile

archive = (
    "/content/drive/MyDrive/"
    "mm-dg-healthcare-data/ptbxl.tar"
)
target = "/content/mm-dg-healthcare/data/raw/ptbxl"

assert os.path.isfile(archive), f"Archive not found: {archive}"

os.makedirs(target, exist_ok=True)

with tarfile.open(archive, "r:*") as tar:
    # Inspect archive paths before extracting.
    members = tar.getmembers()
    print("Archive entries:", len(members))
    print("First few paths:")
    for member in members[:10]:
        print(member.name)

    # Reject absolute paths and path traversal.
    for member in members:
        destination = os.path.realpath(
            os.path.join(target, member.name)
        )
        if os.path.commonpath([target, destination]) != os.path.realpath(target):
            raise ValueError(
                f"Unsafe archive path: {member.name}"
            )

    tar.extractall(target, filter="data")

print("Extraction finished.")

Archive entries: 43648
First few paths:
ptbxl
ptbxl/scp_statements.csv
ptbxl/records100
ptbxl/records100/13000
ptbxl/records100/13000/13674_lr.dat
ptbxl/records100/13000/13541_lr.hea
ptbxl/records100/13000/13950_lr.dat
ptbxl/records100/13000/13065_lr.dat
ptbxl/records100/13000/13390_lr.hea
ptbxl/records100/13000/13239_lr.hea
Extraction finished.


In [5]:
import os

target = "/content/mm-dg-healthcare/data/raw/ptbxl"

for root, dirs, files in os.walk(target):
    for filename in files:
        if filename in ("ptbxl_database.csv", "scp_statements.csv"):
            print("Found:", os.path.join(root, filename))

database = os.path.join(target, "ptbxl_database.csv")
signals = os.path.join(target, "records100")

print("Database exists:", os.path.isfile(database))
print("Signal folder exists:", os.path.isdir(signals))

assert os.path.isfile(database), "Database CSV not found at expected path."
assert os.path.isdir(signals), "records100 folder not found at expected path."

print("Dataset verification passed.")

Found: /content/mm-dg-healthcare/data/raw/ptbxl/ptbxl/ptbxl_database.csv
Found: /content/mm-dg-healthcare/data/raw/ptbxl/ptbxl/scp_statements.csv
Database exists: False
Signal folder exists: False


AssertionError: Database CSV not found at expected path.

In [6]:
import os

base = "/content/mm-dg-healthcare/data/raw/ptbxl"

for root, dirs, files in os.walk(base):
    depth = root.replace(base, "").count(os.sep)
    if depth <= 2:
        print(root)
        for f in files[:5]:
            print("   ", f)

/content/mm-dg-healthcare/data/raw/ptbxl
/content/mm-dg-healthcare/data/raw/ptbxl/ptbxl
    robots.txt
    ptbxl_database.csv
    scp_statements.csv
/content/mm-dg-healthcare/data/raw/ptbxl/ptbxl/records100
    index.html


In [7]:
import tarfile

archive = (
    "/content/drive/MyDrive/"
    "mm-dg-healthcare-data/ptbxl.tar"
)

with tarfile.open(archive, "r:*") as tar:
    members = tar.getmembers()

    print("Total archive entries:", len(members))
    print("\nFirst 30 archive paths:")

    for member in members[:30]:
        print(member.name, member.size)

Total archive entries: 43648

First 30 archive paths:
ptbxl 0
ptbxl/scp_statements.csv 9720
ptbxl/records100 0
ptbxl/records100/13000 0
ptbxl/records100/13000/13674_lr.dat 24000
ptbxl/records100/13000/13541_lr.hea 609
ptbxl/records100/13000/13950_lr.dat 24000
ptbxl/records100/13000/13065_lr.dat 24000
ptbxl/records100/13000/13390_lr.hea 600
ptbxl/records100/13000/13239_lr.hea 604
ptbxl/records100/13000/13554_lr.hea 601
ptbxl/records100/13000/13274_lr.dat 24000
ptbxl/records100/13000/13260_lr.hea 615
ptbxl/records100/13000/13083_lr.hea 604
ptbxl/records100/13000/13324_lr.hea 605
ptbxl/records100/13000/13790_lr.hea 609
ptbxl/records100/13000/13861_lr.hea 602
ptbxl/records100/13000/13216_lr.dat 24000
ptbxl/records100/13000/13629_lr.hea 594
ptbxl/records100/13000/13581_lr.hea 601
ptbxl/records100/13000/13514_lr.hea 608
ptbxl/records100/13000/13314_lr.hea 603
ptbxl/records100/13000/13236_lr.hea 611
ptbxl/records100/13000/13929_lr.dat 24000
ptbxl/records100/13000/13630_lr.dat 24000
ptbxl/reco

In [8]:
import os
import tarfile
import shutil

archive = (
    "/content/drive/MyDrive/"
    "mm-dg-healthcare-data/ptbxl.tar"
)

base = "/content/mm-dg-healthcare/data/raw/ptbxl"
temp = "/content/ptbxl_restore_temp"

assert os.path.isfile(archive), "PTB-XL archive missing."

if os.path.exists(temp):
    shutil.rmtree(temp)
os.makedirs(temp, exist_ok=True)

with tarfile.open(archive, "r:*") as tar:
    # Extract only the expected top-level PTB-XL directory.
    members = [
        m for m in tar.getmembers()
        if m.name == "ptbxl" or m.name.startswith("ptbxl/")
    ]

    assert members, "No ptbxl directory found in archive."

    for member in members:
        destination = os.path.realpath(
            os.path.join(temp, member.name)
        )
        if os.path.commonpath([temp, destination]) != os.path.realpath(temp):
            raise ValueError(f"Unsafe archive path: {member.name}")

    tar.extractall(temp, members=members, filter="data")

restored = os.path.join(temp, "ptbxl")

print("Database:", os.path.isfile(
    os.path.join(restored, "ptbxl_database.csv")
))
print("SCP statements:", os.path.isfile(
    os.path.join(restored, "scp_statements.csv")
))
print("ECG record directory:", os.path.isdir(
    os.path.join(restored, "records100")
))

print("Example record files:")
for root, dirs, files in os.walk(
    os.path.join(restored, "records100")
):
    for name in files[:5]:
        print(os.path.join(root, name))
    if files:
        break

Database: True
SCP statements: True
ECG record directory: True
Example record files:
/content/ptbxl_restore_temp/ptbxl/records100/index.html


In [9]:
import tarfile

archive = (
    "/content/drive/MyDrive/"
    "mm-dg-healthcare-data/ptbxl.tar"
)

with tarfile.open(archive, "r:*") as tar:
    members = tar.getmembers()

    # Inspect a known record from the earlier archive listing
    for name in [
        "ptbxl/records100/13000/13674_lr.dat",
        "ptbxl/records100/13000/13541_lr.hea",
    ]:
        member = tar.getmember(name)
        print("\nFile:", member.name)
        print("Size:", member.size)

        f = tar.extractfile(member)
        print("First bytes:", repr(f.read(100)))


File: ptbxl/records100/13000/13674_lr.dat
Size: 24000
First bytes: b'g\xff>\x00\xd7\x00-\x00H\xff\x8a\x00\xbb\x00\n\x01\xe8\x016\x01\x0f\x00~\xffn\xffL\x00\xde\x00#\x00H\xff\x95\x00\xc8\x00*\x01\x13\x02k\x01\x11\x00\x81\xffw\xffa\x00\xea\x00\x14\x00F\xff\xa5\x00\xcf\x00[\x01T\x02\xb4\x01\x16\x00\x88\xff\x8b\xffq\x00\xe5\x00\x02\x00S\xff\xab\x00\xce\x00v\x01\x82\x02\xff\x01\x19\x00\x8d\xff\xa5\xffy\x00'

File: ptbxl/records100/13000/13541_lr.hea
Size: 609
First bytes: b'13541_lr 12 100 1000\n13541_lr.dat 16 1000.0(0)/mV 16 0 -31 62253 0 I\n13541_lr.dat 16 1000.0(0)/mV 16'


In [10]:
import os

root = "/content/ptbxl_restore_temp/ptbxl/records100"

for current, dirs, files in os.walk(root):
    matches = [
        f for f in files
        if f.endswith(".hea") or f.endswith(".dat")
    ]
    if matches:
        print("Found signal files in:", current)
        print(matches[:10])
        break
else:
    print("No .hea or .dat files found in extracted records100.")

Found signal files in: /content/ptbxl_restore_temp/ptbxl/records100/09000
['09965_lr.dat', '09234_lr.dat', '09816_lr.hea', '09398_lr.hea', '09962_lr.dat', '09649_lr.hea', '09128_lr.dat', '09514_lr.hea', '09066_lr.hea', '09759_lr.hea']


In [11]:
import os
import shutil

base = "/content/mm-dg-healthcare/data/raw/ptbxl"
restored = "/content/ptbxl_restore_temp/ptbxl"
old = base + "_invalid_backup"

# Verify real data before changing anything.
assert os.path.isfile(os.path.join(restored, "ptbxl_database.csv"))
assert os.path.isfile(os.path.join(restored, "scp_statements.csv"))

signal_root = os.path.join(restored, "records100")
assert os.path.isdir(signal_root)

# Ensure real ECG files are present.
hea_count = 0
dat_count = 0

for root, dirs, files in os.walk(signal_root):
    hea_count += sum(f.endswith(".hea") for f in files)
    dat_count += sum(f.endswith(".dat") for f in files)

print("Header files:", hea_count)
print("Signal files:", dat_count)

assert hea_count > 0 and dat_count > 0

# Preserve the old extraction; don't overwrite an existing backup.
if os.path.exists(old):
    raise FileExistsError(
        f"Backup already exists: {old}. Stop and inspect it first."
    )

if os.path.exists(base):
    os.rename(base, old)

shutil.copytree(restored, base)

print("Verified dataset installed at:", base)

Header files: 21799
Signal files: 21799
Verified dataset installed at: /content/mm-dg-healthcare/data/raw/ptbxl


In [12]:
import os

base = "/content/mm-dg-healthcare/data/raw/ptbxl"

assert os.path.isfile(os.path.join(base, "ptbxl_database.csv"))
assert os.path.isfile(os.path.join(base, "scp_statements.csv"))
assert os.path.isdir(os.path.join(base, "records100"))

hea_count = sum(
    f.endswith(".hea")
    for root, dirs, files in os.walk(os.path.join(base, "records100"))
    for f in files
)
dat_count = sum(
    f.endswith(".dat")
    for root, dirs, files in os.walk(os.path.join(base, "records100"))
    for f in files
)

print("Database: OK")
print("SCP statements: OK")
print("ECG header files:", hea_count)
print("ECG signal files:", dat_count)

Database: OK
SCP statements: OK
ECG header files: 21799
ECG signal files: 21799


## Configuration Validation

Verify the fusion experiment configuration before training. This run combines ECG signals with age/sex metadata, trains on site 1, and evaluates on held-out site 2. The HYP class is the selected rare class. Modality dropout remains disabled for this initial experiment.

In [13]:
%cd /content/mm-dg-healthcare

import os
import yaml
import torch

with open("configs/fusion_metadata.yaml", "r") as f:
    cfg = yaml.safe_load(f)

print("GPU:", torch.cuda.get_device_name(0))
print("Dataset path:", cfg["data"]["ptbxl_dir"])
print("Train sites:", cfg["data"]["train_sites"])
print("Test sites:", cfg["data"]["test_sites"])
print("Epochs:", cfg["train"]["epochs"])
print("Rare class:", cfg["rare_class"]["name"])
print("Modality dropout:", cfg["modality_dropout"]["enabled"])
print("Output directory:", cfg["output"]["run_dir"])

data_dir = cfg["data"]["ptbxl_dir"]
assert os.path.isfile(os.path.join(data_dir, "ptbxl_database.csv"))
assert os.path.isfile(os.path.join(data_dir, "scp_statements.csv"))
assert os.path.isdir(os.path.join(data_dir, "records100"))
assert cfg["data"]["train_sites"] == [1]
assert cfg["data"]["test_sites"] == [2]
assert cfg["modality_dropout"]["enabled"] is False

print("\nConfiguration validation passed.")

/content
GPU: Tesla T4
Dataset path: data/raw/ptbxl
Train sites: [1]
Test sites: [2]
Epochs: 20
Rare class: HYP
Modality dropout: False
Output directory: experiments/results/fusion_metadata_site1v2

Configuration validation passed.


In [14]:
run_dir = cfg["output"]["run_dir"]

print("Output directory already exists:", os.path.exists(run_dir))

if os.path.exists(run_dir):
    print("Existing files:", os.listdir(run_dir))

Output directory already exists: False


## Train the ECG + Metadata Fusion Model

Train the fusion model using the validated configuration. The experiment will write its outputs to `experiments/results/fusion_metadata_site1v2/`, keeping it separate from the existing ECG-only baseline.

This experiment evaluates metadata fusion; missing-modality robustness will be tested separately.

In [16]:
%pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 15.2 MB/s eta 0:00:00


In [17]:
import wfdb
import torch

print("WFDB version:", wfdb.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

WFDB version: 4.3.1
GPU available: True
GPU: Tesla T4


In [18]:
%cd /content/mm-dg-healthcare

!python -m src.training.trainer_fusion --config configs/fusion_metadata.yaml

/content
Device: cuda
epoch 1/20 loss=1.1447 val_rare_f1=0.1750 val_macro_f1=0.4585
epoch 2/20 loss=0.9135 val_rare_f1=0.0741 val_macro_f1=0.4099
epoch 3/20 loss=0.8353 val_rare_f1=0.3093 val_macro_f1=0.5274
epoch 4/20 loss=0.7921 val_rare_f1=0.2913 val_macro_f1=0.5127
epoch 5/20 loss=0.7551 val_rare_f1=0.4030 val_macro_f1=0.5301
epoch 6/20 loss=0.7200 val_rare_f1=0.3770 val_macro_f1=0.5211
epoch 7/20 loss=0.6936 val_rare_f1=0.3358 val_macro_f1=0.5292
epoch 8/20 loss=0.6724 val_rare_f1=0.4430 val_macro_f1=0.5691
epoch 9/20 loss=0.6493 val_rare_f1=0.3540 val_macro_f1=0.5356
epoch 10/20 loss=0.6048 val_rare_f1=0.1558 val_macro_f1=0.4930
epoch 11/20 loss=0.5907 val_rare_f1=0.3019 val_macro_f1=0.5170
epoch 12/20 loss=0.5479 val_rare_f1=0.3942 val_macro_f1=0.5583
epoch 13/20 loss=0.5185 val_rare_f1=0.4000 val_macro_f1=0.5557
epoch 14/20 loss=0.4907 val_rare_f1=0.3243 val_macro_f1=0.5306
epoch 15/20 loss=0.4632 val_rare_f1=0.3167 val_macro_f1=0.5307
epoch 16/20 loss=0.4311 val_rare_f1=0.3535

In [19]:
import os
import shutil

src = (
    "/content/mm-dg-healthcare/"
    "experiments/results/fusion_metadata_site1v2"
)
backup_root = (
    "/content/drive/MyDrive/"
    "mm-dg-healthcare-data/results_backup"
)
dst = os.path.join(backup_root, "fusion_metadata_site1v2")

assert os.path.isdir(src), f"Results directory missing: {src}"

os.makedirs(backup_root, exist_ok=True)

print("Experiment files:")
for name in sorted(os.listdir(src)):
    path = os.path.join(src, name)
    print(name, "—", "directory" if os.path.isdir(path) else
          f"{os.path.getsize(path) / 1024:.1f} KB")

if os.path.exists(dst):
    print("\nA Drive backup already exists at:", dst)
    print("Do not overwrite it blindly. Inspect it before proceeding.")
else:
    shutil.copytree(src, dst)
    print("\nBackup copied to:", dst)

    print("Backup verified:", os.path.isdir(dst))
    print("Backed-up files:", sorted(os.listdir(dst)))

Experiment files:
best_model.pt — 908.5 KB
config_used.yaml — 0.5 KB
history.json — 11.5 KB
metadata_normalization.json — 0.2 KB
metrics.json — 2.6 KB

Backup copied to: /content/drive/MyDrive/mm-dg-healthcare-data/results_backup/fusion_metadata_site1v2
Backup verified: True
Backed-up files: ['best_model.pt', 'config_used.yaml', 'history.json', 'metadata_normalization.json', 'metrics.json']


## Interpretation checklist
- Did the fusion model improve HYP F1 compared with the signal-only condition?
- Did metadata alone perform strongly? If yes, fusion gains need cautious interpretation.
- Site 2 was not used for checkpoint selection.
- This is ECG + demographic metadata, not independent physician-report fusion; update the proposal framing if this becomes the primary modality pair.